In [0]:
import sys
sys.path.append('/Workspace/Users/saik84328@gmail.com')

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from SetUp.Config import bronze_schema, silver_schema, gold_schema
from pyspark.sql.types import *
from datetime import datetime
import uuid


In [0]:
start_time = datetime.now()

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/ValidationFramework

In [0]:
patient_df = spark.table("helathcare_silver.helathcare_silver.sl_patient")
display(patient_df)

In [0]:
Claims_df = spark.table("helathcare_silver.helathcare_silver.SL_ClaimsData")
display(Claims_df)

In [0]:
hospial_df = spark.table("helathcare_silver.helathcare_silver.SL_Hosptial")
display(hospial_df)

In [0]:

insurance_df = spark.table("helathcare_silver.helathcare_silver.SL_insurancedeatils")
display(insurance_df)

In [0]:


Joined_df = F.broadcast(
    patient_df.alias("p")
).join(
    
    Claims_df.alias("c"),
    
    F.col("p.Patient_id") == F.col("c.Patientid"),
    
    "inner"
).join(
        F.broadcast(hospial_df.alias("d")), 
     
        F.col("c.Placeofservice") == F.col("d.patient_id"),
        
        "inner"
    ).join(
    F.broadcast(insurance_df.alias("i")),
    
    F.col("c.Patientid") == F.col("i.Patient_id"),
    
    "inner")
display(Joined_df)

In [0]:
Joined_df.printSchema()

In [0]:

Patientinfo_df=Joined_df.select(patient_df.Patient_id,patient_df.Birthdate,patient_df.Deathdate,patient_df.Prefix,patient_df.Name.alias("PatientName"),patient_df.Marital,patient_df.Race,patient_df.Gender,patient_df.Birthplace,patient_df.Address,patient_df.City,patient_df.State,patient_df.Country,patient_df.Healthcare_Expenses,patient_df.Healthcare_Coverage,patient_df.Income,Claims_df.Claimid,Claims_df.Type,Claims_df.Cliams_amount,Claims_df.Method,Claims_df.Fromdate,Claims_df.Todate,Claims_df.Placeofservice,Claims_df.Procdurecode,Claims_df.Units,Claims_df.Departmentid,Claims_df.Notes,hospial_df.Hosiptalname,hospial_df.Address.alias("HospitalAdress") ,hospial_df.City.alias("HospitalCity"),hospial_df.State.alias("HospitalState"),hospial_df.Phone.alias("HospitalPhone"),hospial_df.Revenue.alias("HospitalRevenue"),hospial_df.Utilization.alias("HospitalUtilization"),insurance_df.Insurance_provider,insurance_df.Organization_name,insurance_df.Amount_covered,insurance_df.Amount_uncovered,insurance_df.Revenue
,insurance_df.Covered_encounters,insurance_df.Uncovered_encounters,insurance_df.Covered_medications,insurance_df.Uncovered_medications,insurance_df.Covered_procedures,insurance_df.Uncovered_procedures,insurance_df.Covered_immunizations,insurance_df.Uncovered_immunizations,insurance_df.Unique_customers,insurance_df.Member_months,insurance_df.Member_id)
display(Patientinfo_df)


In [0]:
# spark.sql(f"DROP TABLE IF EXISTS helathcare_silver.helathcare_silver.SL_PatienatInfo")

# Patientinfo_df.write.format("delta") \
#     .mode("append") \
#     .partitionBy("BirthDate") \
#     .saveAsTable(
#         "helathcare_silver.helathcare_silver.SL_PatienatInfo"
#     )

In [0]:
#create tempview
Patientinfo_df.createOrReplaceTempView(
    "Patientinfo"
)


In [0]:
%sql
MERGE INTO helathcare_silver.helathcare_silver.SL_PatienatInfo tgt
USING Patientinfo src

ON tgt.Patient_id = src.Patient_id

WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:

GoldPatientdf=spark.table("helathcare_silver.helathcare_silver.SL_PatienatInfo")
GoldPatientdf=GoldPatientdf.count()
print(GoldPatientdf)


In [0]:
end_time = datetime.now()

duration_seconds = int(
    (end_time - start_time).total_seconds()
)

print(duration_seconds)

In [0]:
from pyspark.sql.types import LongType
from datetime import datetime

# Get Workflow Run ID
try:
    run_id = dbutils.jobs.taskContext().taskRunId()
except:
    run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d%H%M%S')}"

target_table = "helathcare_silver.helathcare_silver.SL_PatienatInfo"

# Get metadata
notebook_name, table_name, layer = get_audit_metadata(target_table)

status = "SUCCESS"
error_message = None
record_count = 0

In [0]:
end_time = datetime.now()

write_audit(
    target_table=target_table,
    run_id=run_id,
    record_count=GoldPatientdf,
    start_time=start_time,
    end_time=end_time,
    status=status,
    error_message=error_message
)